# Meet AlexNet: loading a network and classifying one image
**Background:** Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *NeurIPS, 25*.

AlexNet was trained on **ImageNet**: about 1.2 million photos, each labelled with one of **1,000 categories**.

**What you will do**
1. Load AlexNet with its trained weights
2. Print its architecture and look at its layers
3. Prepare one image for the network
4. Classify the image and look at the top-5 predictions
5. See what AlexNet does with things it has **no label** for

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torchvision import models, transforms
device = "cpu"

In [ ]:
# Find the course data: on JupyterHub it is in Sciebo, on your own computer
# it is in the "images" folder next to this notebook
sciebo = sorted(Path.home().glob("sciebo/*/Week2/"))     # works even if the shared folder was renamed
if sciebo:
    DATA = sciebo[0]                                     # JupyterHub (Sciebo)
    IMAGE_FILE = DATA / "example.JPEG"
else:
    DATA = Path.cwd()                                    # your computer: the folder of this notebook
    IMAGE_FILE = next((DATA / "images").glob("example.*"), DATA / "images" / "example.JPEG")

if not IMAGE_FILE.exists():
    raise FileNotFoundError(
        f"The image example.JPEG (or .jpg) was not found in {IMAGE_FILE.parent}.\n"
        "On JupyterHub: did you connect Sciebo when starting the server, and accept the shared CNU folder?\n"
        "On your own computer: put example.JPEG (or example.jpg) in the \"images\" folder next to this notebook."
    )
print(f"Using data from {DATA}")

## 1. Load AlexNet
The **architecture** is the structure of the network: which layers there are and how they are connected. The **weights** are the numbers the network learned during training. Without trained weights, AlexNet would just guess randomly.

In [ ]:
# Load AlexNet with its trained weights.
weights = models.AlexNet_Weights.IMAGENET1K_V1   # the ImageNet-trained version
model = models.alexnet(weights=weights)

# .eval() switches off training-only behavior, so predictions are stable
model = model.to(device).eval()

# The names of the 1,000 categories, in the order of the network's outputs
categories = weights.meta["categories"]
print(f"AlexNet knows {len(categories)} categories")
print(categories)

## 2. The architecture

In [ ]:
print(model)

AlexNet has two parts:

- **`features`**: 5 **convolutional layers** (`Conv2d`). Each one slides small filters over the image and detects patterns: edges and colours in the first layer, more complex shapes and object parts in later layers. After each one comes a **`ReLU`**, which sets negative values to zero, and sometimes a **`MaxPool2d`**, which shrinks the image by keeping the strongest response in each small region.
- **`classifier`**: 3 **fully connected layers** (`Linear`). These combine all the features into one decision. The last layer has **1,000 outputs**, one for each category. (`Dropout` is only used during training.)

In [ ]:
# Look at a single layer: the first convolutional layer
first_layer = model.features[0]
print(first_layer)

## 3. Prepare one image
AlexNet always takes a 224 × 224 pixel image. Every image is **resized**, **cropped** to the centre, and its colours are **normalised** the same way as during training.

In [ ]:
img = Image.open(IMAGE_FILE).convert("RGB")       # convert("RGB") ensures 3 colour channels

# The full preparation that goes into the network
preprocess = transforms.Compose([
    transforms.Resize(256),        # resize
    transforms.CenterCrop(224),    # cut out the central 224 x 224 square
    transforms.ToTensor(),         # turn the image into numbers (PyTorch's tensor format)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # same colour normalisation
                         std=[0.229, 0.224, 0.225]),   # as in ImageNet training
])

# The same steps without the number conversion, so we can SHOW what the network sees
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img); axes[0].set_title(f"original {img.size[0]} x {img.size[1]}")
axes[1].imshow(what_network_sees(img)); axes[1].set_title("network input 224 x 224")
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
# The network expects a BATCH of images, so we add a batch dimension of size 1
x = preprocess(img).unsqueeze(0).to(device)
print(f"Input shape: {tuple(x.shape)}   (images, colour channels, height, width)")

## 4. Classify the image

In [ ]:
# We only predict, we don't train: no gradients needed
with torch.no_grad():
    outputs = model(x)

print(f"Output shape: {tuple(outputs.shape)}   (images, categories)")

# Softmax turns the 1,000 raw outputs into probabilities that sum to 1
probs = torch.softmax(outputs, dim=1)[0]
print(f"Sum of all probabilities: {probs.sum().item():.2f}")

In [ ]:
# The 5 most likely categories
top_probs, top_idx = probs.topk(5)
for p, i in zip(top_probs, top_idx):
    print(f"{categories[i]:30s} {p.item():.3f}")

**Why are some probabilities 0.000?** This is a rounding effect. `:.3f` prints only three decimals, so anything below 0.0005 shows as `0.000`. If AlexNet is very confident about its first guess (e.g. 0.998), the other 999 categories share what is left, so guesses 2–5 can be tiny.

The top 5 is still meaningful: `topk` sorts from highest to lowest, so the guesses that show `0.000` are still ranked by their real values.

### Congrats! You classified an image using a CNN!!!

## 5. Things AlexNet has no label for
None of these objects has a label among AlexNet's 1,000 categories. But AlexNet **has to** answer since there is no "I don't know" option.

#### Q1. AlexNet has no label for giraffes or helicopters, so it has to pick something else. Do its guesses make sense? Is there anything common between the 'true' label of the image and the 'predicted' label? What is the commonality?

In [ ]:
IMAGE_DIR = IMAGE_FILE.parent   # the same folder as example.jpg
unknown = {}    # unknown[name] = image, e.g. unknown["giraffe"]

for path in sorted(IMAGE_DIR.glob("unknown_*")):    # e.g. unknown_giraffe.jpg
    name = path.stem.partition("_")[2]                 # "unknown_giraffe" -> "giraffe"
    unknown[name] = Image.open(path).convert("RGB")

print(f"Unknown things: {list(unknown)}")

We set up two functions: one classifies several images at once, the other shows each image next to its top-5 guesses.

In [ ]:
def classify(imgs, k=5):
    """Return the k most likely labels and their probabilities for each image."""
    # Prepare all images and stack them into one batch
    batch = torch.stack([preprocess(img) for img in imgs]).to(device)

    # We only predict, we don't train: no gradients needed
    with torch.no_grad():
        outputs = model(batch)                    # shape: (number of images, 1000)

    # Softmax turns the 1,000 raw outputs into probabilities that sum to 1
    probs = torch.softmax(outputs, dim=1)

    # Keep the k highest probabilities and their category numbers
    top_probs, top_idx = probs.topk(k, dim=1)

    results = []
    for p_row, i_row in zip(top_probs, top_idx):
        results.append([(categories[i], p.item()) for p, i in zip(p_row, i_row)])
    return results

def show_top5(images, names, results, title=None):
    """One row per image: the image on the left, its top-5 predictions on the right."""
    n = len(images)
    fig, axes = plt.subplots(n, 2, figsize=(9, 2.3 * n), gridspec_kw={"width_ratios": [1, 1.7]})
    axes = np.atleast_2d(axes)
    for row, (img, name, res) in enumerate(zip(images, names, results)):
        ax_img, ax_bar = axes[row]
        ax_img.imshow(what_network_sees(img)); ax_img.axis("off")
        ax_img.set_title(name, fontsize=10)

        labels = [label for label, _ in res][::-1]   # reversed, so the top guess is on top
        probs  = [p for _, p in res][::-1]
        ax_bar.barh(labels, probs, color="tab:blue")
        ax_bar.set_xlim(0, 1)
        for y, p in enumerate(probs):
            ax_bar.text(p + 0.01, y, f"{p:.2f}", va="center", fontsize=9)
    axes[-1, 1].set_xlabel("probability")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

In [ ]:
unknown_names = list(unknown)
unknown_imgs = [unknown[name] for name in unknown_names]
unknown_results = classify(unknown_imgs)
show_top5(unknown_imgs, [f"unknown: {n}" for n in unknown_names], unknown_results,
          title="Things AlexNet has no label for")